# 🛰️ SatQuery AI — VLM Training Data Pipeline (BigEarthNet & SARLANG-1M Multi-Session)

**Problem Statement ID:** 26167 (ISRO / SAC)  
**Input Datasets:**  
- `yugkitatti`: BigEarthNet base (`BigEarthNet-S1.tar.zst`, `BigEarthNet-S2.tar.zst`, `metadata.parquet`, `metadata_for_patches_with_snow_cloud_or_shadow.parquet`)  
- `YiminJimmy/SARLANG-1M`: Hugging Face (`SARimages_preprocessed.zip` + `Text.zip`)  

---
### ⚡ Shard Sizing & Disk Safeguard Architecture:
- **BigEarthNet Shards: 10.6 GB per Shard (50/50 S1 & S2 split):**  
  - **5.3 GB Sentinel-1 SAR** (VV/VH dual-pol radar backscatter)  
  - **5.3 GB Sentinel-2 Optical** (B02, B03, B04, B08 10m VNIR bands)  
  - Sized at ~30,000 paired scenes per shard (~360 KB uncompressed per scene) to guarantee exactly ~10.6 GB footprint in `/tmp/fusion_data`.  
- **SARLANG-1M Shards: 7.1 GB per Shard:**  
  - Partitioned dynamically into ~7.1 GB shards of preprocessed SAR images and conversation annotations (`sarlang_shard_00000.jsonl`, `sarlang_shard_00001.jsonl`, ...).  
- **Proven Stream-Extract-Then-Wipe Pipeline:**  
  - Reuses `satquery_6shard_pipeline.ipynb` `tar.zst` streaming decompression (`zstd.ZstdDecompressor` + `tarfile.open(mode='r|')`). The 110 GB input archive remains untouched on the Kaggle input mount.  
  - Local scratch write stays strictly within the current shard footprint (~10.6 GB in `/tmp/fusion_data`), wiped after building each shard.  
- **Deterministic Global Sharding:** Seed `42` random shuffle aligns global shard boundaries 1:1 with SAR ResNet-50 / ConvNeXt training.  
- **8.5-Hour Session Guard (`MAX_RUNTIME_HOURS = 8.5`):** Shard loop checks elapsed time before each shard and exits cleanly with progress saved in `_build_progress.json`.


In [ ]:
# 1. Install dependencies
!pip install -q zstandard pyarrow huggingface_hub pillow


In [ ]:
# 2. Paths, hyperparameters, global sharding (mirrors cell 3 of the SAR notebook)
import os, sys, time, json, glob, shutil
import numpy as np
import pandas as pd

# --- Wall-Clock Runtime Safeguard (same value as the SAR notebook) ---
MAX_RUNTIME_HOURS = 8.5
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600
SESSION_START_TIME = time.time()

# --- Sharding Specifications ---
# BigEarthNet: exactly 10.6 GB per shard (half S1 = 5.3 GB, half S2 = 5.3 GB)
TARGET_BIGEARTH_SHARD_GB = 10.6
TARGET_S1_GB_PER_SHARD = 5.3
TARGET_S2_GB_PER_SHARD = 5.3
TARGET_S1_BYTES_PER_SHARD = int(TARGET_S1_GB_PER_SHARD * (1024**3))
TARGET_S2_BYTES_PER_SHARD = int(TARGET_S2_GB_PER_SHARD * (1024**3))
TARGET_BIGEARTH_TOTAL_BYTES = int(TARGET_BIGEARTH_SHARD_GB * (1024**3))

# SARLANG-1M: exactly 7.1 GB per shard
TARGET_SARLANG_SHARD_GB = 7.1
TARGET_SARLANG_SHARD_BYTES = int(TARGET_SARLANG_SHARD_GB * (1024**3))

# Footprint math:
# Uncompressed per paired scene: ~180 KB S1 (VV+VH) + ~180 KB S2 (B02,B03,B04,B08) ≈ 360 KB/scene.
# 10.6 GB (5.3 GB S1 + 5.3 GB S2) corresponds to ~30,000 paired scenes per shard.
SAMPLES_PER_SHARD = 30000
SHARDS_PER_SESSION = 6
MATCHED_PER_PATCH = 1
MISMATCHED_PER_PATCH = 1

# --- Locate BigEarthNet Datasets in /kaggle/input (identical to the SAR notebook) ---
print("locating BigEarthNet-S1 and S2 archives in /kaggle/input...")
s1_archives = glob.glob("/kaggle/input/**/BigEarthNet-S1.tar.zst", recursive=True)
s2_archives = glob.glob("/kaggle/input/**/BigEarthNet-S2.tar.zst", recursive=True)
if not s1_archives or not s2_archives:
    raise FileNotFoundError("Both BigEarthNet-S1.tar.zst and BigEarthNet-S2.tar.zst must be attached!")

S1_ARCHIVE = s1_archives[0]
S2_ARCHIVE = s2_archives[0]
DATASET_ROOT = os.path.dirname(S1_ARCHIVE)
METADATA_PATH = os.path.join(DATASET_ROOT, "metadata.parquet")
METADATA_EXCLUDED_PATH = os.path.join(DATASET_ROOT, "metadata_for_patches_with_snow_cloud_or_shadow.parquet")
print(f"S1 Archive: {S1_ARCHIVE}")
print(f"S2 Archive: {S2_ARCHIVE}")

OUTPUT_DIR = "/kaggle/working"
TMP_FUSION_DIR = "/tmp/fusion_data"
S1_EXTRACT_DIR = os.path.join(TMP_FUSION_DIR, "s1")
S2_EXTRACT_DIR = os.path.join(TMP_FUSION_DIR, "s2")
DATA_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "vlm_verification_data")
PROGRESS_PATH = os.path.join(DATA_OUTPUT_DIR, "_build_progress.json")
os.makedirs(S1_EXTRACT_DIR, exist_ok=True)
os.makedirs(S2_EXTRACT_DIR, exist_ok=True)
os.makedirs(DATA_OUTPUT_DIR, exist_ok=True)

CORINE_19_CLASSES = [
    'Agro-forestry areas', 'Arable land', 'Beaches, dunes, sands', 'Broad-leaved forest',
    'Coastal wetlands', 'Complex cultivation patterns', 'Coniferous forest',
    'Industrial or commercial units', 'Inland waters', 'Inland wetlands',
    'Land principally occupied by agriculture, with significant areas of natural vegetation',
    'Marine waters', 'Mixed forest', 'Moors, heathland and sclerophyllous vegetation',
    'Natural grassland and sparsely vegetated areas', 'Pastures', 'Permanent crops',
    'Transitional woodland, shrub', 'Urban fabric'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CORINE_19_CLASSES)}
print(f"Initialized BigEarthNet sharding: {TARGET_BIGEARTH_SHARD_GB} GB/shard ({TARGET_S1_GB_PER_SHARD} GB S1 + {TARGET_S2_GB_PER_SHARD} GB S2, {SAMPLES_PER_SHARD:,} scenes/shard)")
print(f"Initialized SARLANG-1M sharding: {TARGET_SARLANG_SHARD_GB} GB/shard")


## Smart auto-detection of previously-built VLM data

Same idea as `find_and_stage_best_checkpoint` in the SAR notebook -- if you saved
`vlm_verification_data/` (or a zip of it) as a Kaggle Dataset after a prior
session and re-attached it, this stages it so building resumes instead of
restarting.


In [ ]:
def find_and_stage_previous_build():
    if os.path.exists(PROGRESS_PATH):
        print("progress file already staged in /kaggle/working.")
        return

    candidate_zips = glob.glob("/kaggle/input/**/vlm_verification_data*.zip", recursive=True)
    if candidate_zips:
        candidate_zips.sort(key=lambda p: os.path.getmtime(p), reverse=True)
        chosen = candidate_zips[0]
        print(f"found uploaded data-build archive: {chosen} -- unpacking into {DATA_OUTPUT_DIR}...")
        try:
            import zipfile
            with zipfile.ZipFile(chosen, 'r') as zf:
                zf.extractall(DATA_OUTPUT_DIR)
            print("unpacked previous build artifacts.")
            return
        except Exception as e:
            print(f"could not unpack {chosen}: {e}")

    candidate_progress = glob.glob("/kaggle/input/**/_build_progress.json", recursive=True)
    if candidate_progress:
        src_dir = os.path.dirname(candidate_progress[0])
        print(f"found staged build directory at {src_dir} -- copying shard files...")
        for f in glob.glob(os.path.join(src_dir, "*.jsonl")):
            shutil.copy(f, DATA_OUTPUT_DIR)
        shutil.copy(candidate_progress[0], PROGRESS_PATH)
        print(f"staged {len(glob.glob(os.path.join(DATA_OUTPUT_DIR, '*.jsonl')))} shard file(s).")
        return

    print("no previous VLM data build found in /kaggle/input -- starting fresh.")

find_and_stage_previous_build()


## Metadata ingestion (identical to the SAR notebook's cell 3)

Same cloud/shadow-recovery merge, same snow filter, same deterministic seed-42
shuffle -- this guarantees the shard boundaries here line up 1:1 with the SAR
notebook's shards, which matters if you ever want to cross-reference "this VLM
example came from the same global shard as this SAR training batch."


In [ ]:
print("ingesting dataset metadata to construct continuous global sharding index...")
meta_df = pd.read_parquet(METADATA_PATH)

if os.path.exists(METADATA_EXCLUDED_PATH):
    excluded_df = pd.read_parquet(METADATA_EXCLUDED_PATH)
    cloud_recoverable = excluded_df[(~excluded_df['contains_seasonal_snow']) & (excluded_df['contains_cloud_or_shadow'])].copy()
    print(f"including {len(cloud_recoverable):,} cloud/shadow patches -- same recovery logic as SAR notebook")
    meta_df = pd.concat([meta_df, cloud_recoverable], ignore_index=True)

if 'contains_seasonal_snow' in meta_df.columns:
    meta_df = meta_df[~meta_df['contains_seasonal_snow']]

paired_df = meta_df[meta_df['patch_id'].notna() & meta_df['s1_name'].notna()].copy()
print(f"total clean paired candidates: {len(paired_df):,}")

shuffled_all = paired_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
TOTAL_GLOBAL_SHARDS = int(np.ceil(len(shuffled_all) / SAMPLES_PER_SHARD))
print(f"partitioned into {TOTAL_GLOBAL_SHARDS} global shards of up to {SAMPLES_PER_SHARD:,} pairs (~{TARGET_BIGEARTH_SHARD_GB} GB uncompressed: {TARGET_S1_GB_PER_SHARD} GB S1 + {TARGET_S2_GB_PER_SHARD} GB S2) each")

shards_meta = []
for s_idx in range(TOTAL_GLOBAL_SHARDS):
    start_i = s_idx * SAMPLES_PER_SHARD
    end_i = min((s_idx + 1) * SAMPLES_PER_SHARD, len(shuffled_all))
    shards_meta.append(shuffled_all.iloc[start_i:end_i].copy().reset_index(drop=True))


## Disk-safe extraction engine (reused unchanged from the SAR notebook)

Same tar.zst streaming, same VV/VH and B02/B03/B04/B08 band filtering, same
early-exit-on-full-match, same single-clear-per-shard discipline.


In [ ]:
import tarfile
import zstandard as zstd

def clear_fusion_tmp_dir():
    if os.path.exists(TMP_FUSION_DIR):
        shutil.rmtree(TMP_FUSION_DIR, ignore_errors=True)
    os.makedirs(S1_EXTRACT_DIR, exist_ok=True)
    os.makedirs(S2_EXTRACT_DIR, exist_ok=True)

def extract_shard_data(shard_df, shard_num, total_shards=TOTAL_GLOBAL_SHARDS):
    if len(shard_df) == 0:
        print(f"shard {shard_num} contains 0 samples -- skipping extraction.")
        return

    done_flag = os.path.join(TMP_FUSION_DIR, f".shard_{shard_num}_done")
    if os.path.exists(done_flag):
        print(f"shard {shard_num} data already extracted in /tmp/fusion_data.")
        return

    clear_fusion_tmp_dir()
    target_s1_names = set(shard_df['s1_name'].astype(str))
    target_s2_names = set(shard_df['patch_id'].astype(str))

    print(f"\n{'='*70}")
    print(f"📦 [Global Shard {shard_num}/{total_shards}] Target footprint: {TARGET_BIGEARTH_SHARD_GB} GB ({TARGET_S1_GB_PER_SHARD} GB S1 + {TARGET_S2_GB_PER_SHARD} GB S2)")
    print(f"   Extracting {len(target_s1_names):,} S1 radar patches into {S1_EXTRACT_DIR}...")
    extracted_s1 = 0
    extracted_s1_bytes = 0
    t0 = time.time()
    dctx = zstd.ZstdDecompressor()
    with open(S1_ARCHIVE, 'rb') as fh:
        with dctx.stream_reader(fh) as stream:
            with tarfile.open(mode='r|', fileobj=stream) as tar:
                for member in tar:
                    if not member.isfile() or not member.name.endswith('.tif'):
                        continue
                    fname = os.path.basename(member.name)
                    s1_name = fname.replace("_VV.tif", "").replace("_VH.tif", "")
                    if s1_name in target_s1_names:
                        dest_path = os.path.join(S1_EXTRACT_DIR, fname)
                        if not os.path.exists(dest_path):
                            f = tar.extractfile(member)
                            if f:
                                b = f.read()
                                with open(dest_path, 'wb') as out_f:
                                    out_f.write(b)
                                extracted_s1 += 1
                                extracted_s1_bytes += len(b)
                    if extracted_s1 >= len(target_s1_names) * 2:
                        print(f"   all {extracted_s1:,} S1 bands found -- early exit.")
                        break
    s1_gb = extracted_s1_bytes / (1024**3)
    print(f"   shard {shard_num} S1 extraction complete: {extracted_s1:,} files ({s1_gb:.2f} GB / {TARGET_S1_GB_PER_SHARD} GB target) in {time.time()-t0:.1f}s")

    print(f"   Extracting {len(target_s2_names):,} S2 optical patches into {S2_EXTRACT_DIR}...")
    extracted_s2 = 0
    extracted_s2_bytes = 0
    t0 = time.time()
    s2_bands_keep = ('_B02.tif', '_B03.tif', '_B04.tif', '_B08.tif')
    with open(S2_ARCHIVE, 'rb') as fh:
        with dctx.stream_reader(fh) as stream:
            with tarfile.open(mode='r|', fileobj=stream) as tar:
                for member in tar:
                    if not member.isfile() or not any(member.name.endswith(b) for b in s2_bands_keep):
                        continue
                    fname = os.path.basename(member.name)
                    patch_id = fname.split('_B0')[0] if '_B0' in fname else fname.split('_B')[0]
                    if patch_id in target_s2_names:
                        dest_path = os.path.join(S2_EXTRACT_DIR, fname)
                        if not os.path.exists(dest_path):
                            f = tar.extractfile(member)
                            if f:
                                b = f.read()
                                with open(dest_path, 'wb') as out_f:
                                    out_f.write(b)
                                extracted_s2 += 1
                                extracted_s2_bytes += len(b)
                    if extracted_s2 >= len(target_s2_names) * 4:
                        print(f"   all {extracted_s2:,} S2 bands found -- early exit.")
                        break
    s2_gb = extracted_s2_bytes / (1024**3)
    total_shard_gb = s1_gb + s2_gb
    print(f"   shard {shard_num} S2 extraction complete: {extracted_s2:,} files ({s2_gb:.2f} GB / {TARGET_S2_GB_PER_SHARD} GB target) in {time.time()-t0:.1f}s")
    print(f"✨ Shard {shard_num} uncompressed total: {total_shard_gb:.2f} GB (Target: {TARGET_BIGEARTH_SHARD_GB} GB: {TARGET_S1_GB_PER_SHARD} GB S1 + {TARGET_S2_GB_PER_SHARD} GB S2)")

    with open(done_flag, 'w') as f:
        f.write('done')
    total, used, free = shutil.disk_usage("/tmp")
    print(f"   shard {shard_num} /tmp disk: {used/(1024**3):.1f} GB used, {free/(1024**3):.1f} GB free.")


## Label parsing (identical to `extract_active_indices` in the SAR notebook) + judge-pair generation


In [ ]:
import ast, random

def extract_active_indices(raw_label, class_to_idx, num_classes=19):
    if raw_label is None:
        return []
    if isinstance(raw_label, str):
        raw_label = raw_label.strip()
        if raw_label.startswith('[') and raw_label.endswith(']'):
            try:
                raw_label = ast.literal_eval(raw_label)
            except Exception:
                raw_label = [x.strip(" '") for x in raw_label[1:-1].split(',')]
        else:
            raw_label = [x.strip(" '") for x in raw_label.split(',')]
    active_indices = []
    if hasattr(raw_label, '__iter__'):
        for item in raw_label:
            if isinstance(item, (int, np.integer)):
                if 0 <= int(item) < num_classes:
                    active_indices.append(int(item))
            elif isinstance(item, (str, np.str_)):
                s = str(item).strip()
                if s in class_to_idx:
                    active_indices.append(class_to_idx[s])
    return sorted(list(set(active_indices)))

CONFUSABLE_MAP = {
    "Inland waters": ["Inland wetlands", "Marine waters", "Coastal wetlands"],
    "Inland wetlands": ["Inland waters", "Coastal wetlands", "Natural grassland and sparsely vegetated areas"],
    "Coastal wetlands": ["Inland wetlands", "Marine waters", "Beaches, dunes, sands"],
    "Marine waters": ["Inland waters", "Coastal wetlands"],
    "Beaches, dunes, sands": ["Coastal wetlands", "Natural grassland and sparsely vegetated areas"],
    "Pastures": ["Natural grassland and sparsely vegetated areas", "Arable land", "Complex cultivation patterns"],
    "Natural grassland and sparsely vegetated areas": ["Pastures", "Moors, heathland and sclerophyllous vegetation"],
    "Arable land": ["Complex cultivation patterns", "Pastures",
                    "Land principally occupied by agriculture, with significant areas of natural vegetation"],
    "Complex cultivation patterns": ["Arable land",
                                      "Land principally occupied by agriculture, with significant areas of natural vegetation",
                                      "Agro-forestry areas"],
    "Land principally occupied by agriculture, with significant areas of natural vegetation":
        ["Complex cultivation patterns", "Agro-forestry areas", "Pastures"],
    "Agro-forestry areas": ["Land principally occupied by agriculture, with significant areas of natural vegetation",
                             "Mixed forest", "Permanent crops"],
    "Permanent crops": ["Agro-forestry areas", "Complex cultivation patterns"],
    "Broad-leaved forest": ["Mixed forest", "Transitional woodland, shrub"],
    "Coniferous forest": ["Mixed forest", "Transitional woodland, shrub"],
    "Mixed forest": ["Broad-leaved forest", "Coniferous forest", "Transitional woodland, shrub", "Agro-forestry areas"],
    "Transitional woodland, shrub": ["Mixed forest", "Moors, heathland and sclerophyllous vegetation", "Broad-leaved forest"],
    "Moors, heathland and sclerophyllous vegetation": ["Transitional woodland, shrub",
                                                        "Natural grassland and sparsely vegetated areas"],
    "Urban fabric": ["Industrial or commercial units"],
    "Industrial or commercial units": ["Urban fabric"],
}

def get_confusable(true_label, all_labels):
    candidates = [c for c in CONFUSABLE_MAP.get(true_label, []) if c not in all_labels]
    if not candidates:
        candidates = [c for c in CORINE_19_CLASSES if c not in all_labels]
    return random.choice(candidates)

QUESTION_TEMPLATES = [
    "A land-cover classifier predicts \"{candidate}\" for this SAR tile with confidence {conf:.2f}. "
    "Does the visible SAR evidence support this prediction? Answer with Agree or Disagree, "
    "a confidence score from 0 to 1, and a short justification.",
    "Classifier output: \"{candidate}\" (confidence {conf:.2f}). Verify this against the SAR "
    "imagery shown and explain your reasoning.",
]

def build_example(s1_name, patch_id, true_labels):
    examples = []
    s1_image_path = os.path.join(S1_EXTRACT_DIR, f"{s1_name}_VV.tif")  # despeckle/stretch is a preprocessing
                                                                          # step to apply when loading for training,
                                                                          # not baked into these paths -- TODO(adapt)

    for _ in range(MATCHED_PER_PATCH):
        candidate = random.choice(true_labels)
        conf = float(np.clip(np.random.normal(loc=0.80, scale=0.08), 0.5, 0.99))
        q = random.choice(QUESTION_TEMPLATES).format(candidate=candidate, conf=conf)
        others = [l for l in true_labels if l != candidate]
        extra = f", alongside visible characteristics of {others[0]}" if others else ""
        a = f"Agree. Confidence: {conf:.2f}. The SAR backscatter pattern and texture in this tile are consistent with {candidate}{extra}."
        examples.append({"patch_id": patch_id, "s1_name": s1_name, "image": s1_image_path,
                          "label_verdict": "agree", "candidate_label": candidate, "true_labels": true_labels,
                          "conversations": [{"from": "human", "value": q}, {"from": "gpt", "value": a}]})

    for _ in range(MISMATCHED_PER_PATCH):
        anchor = random.choice(true_labels)
        candidate = get_confusable(anchor, true_labels)
        conf = float(np.clip(np.random.normal(loc=0.55, scale=0.12), 0.2, 0.85))
        q = random.choice(QUESTION_TEMPLATES).format(candidate=candidate, conf=conf)
        true_label = random.choice(true_labels)
        a = (f"Disagree. Confidence: {conf:.2f}. The backscatter intensity and texture pattern in this tile "
             f"more closely match {true_label} rather than {candidate}; the SAR signature expected for "
             f"{candidate} is not clearly present here.")
        examples.append({"patch_id": patch_id, "s1_name": s1_name, "image": s1_image_path,
                          "label_verdict": "disagree", "candidate_label": candidate, "true_labels": true_labels,
                          "conversations": [{"from": "human", "value": q}, {"from": "gpt", "value": a}]})
    return examples


## Resume state + guard (mirrors the SAR notebook's shard/epoch resume + completion guard)


In [ ]:
start_shard_idx = 0
if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH) as f:
        start_shard_idx = json.load(f).get("next_shard_idx", 0)
    print(f"resuming at global shard {start_shard_idx + 1}/{TOTAL_GLOBAL_SHARDS}")
else:
    print("starting fresh at global shard 1")

if start_shard_idx >= TOTAL_GLOBAL_SHARDS:
    print("="*75)
    print(f"ALL {TOTAL_GLOBAL_SHARDS} GLOBAL SHARDS ALREADY BUILT INTO VLM TRAINING DATA.")
    print(f"Checkpoint is at global shard {start_shard_idx}, and {TOTAL_GLOBAL_SHARDS} shards exist in the attached dataset.")
    print("No further building required.")
    print("="*75)
    RUN_BUILD = False
else:
    RUN_BUILD = True
    session_end_shard_idx = min(start_shard_idx + SHARDS_PER_SESSION, TOTAL_GLOBAL_SHARDS)
    print(f"this session will build global shards {start_shard_idx + 1} to {session_end_shard_idx} "
          f"({session_end_shard_idx - start_shard_idx} shard(s))")


## Main build loop


In [ ]:
build_interrupted = False
reason_for_stop = ""

if RUN_BUILD:
    for shard_idx in range(start_shard_idx, session_end_shard_idx):
        elapsed = time.time() - SESSION_START_TIME
        if elapsed >= MAX_RUNTIME_SECONDS:
            reason_for_stop = f"reached {MAX_RUNTIME_HOURS}h session safety limit"
            build_interrupted = True
            break

        shard_df = shards_meta[shard_idx]
        if len(shard_df) == 0:
            print(f"global shard {shard_idx + 1} has 0 samples -- skipping.")
            continue

        print(f"\n{'='*70}\nBUILDING GLOBAL SHARD {shard_idx + 1}/{TOTAL_GLOBAL_SHARDS} ({len(shard_df):,} scenes, ~{TARGET_BIGEARTH_SHARD_GB} GB: {TARGET_S1_GB_PER_SHARD} GB S1 + {TARGET_S2_GB_PER_SHARD} GB S2)\n{'='*70}")
        extract_shard_data(shard_df, shard_idx + 1, total_shards=TOTAL_GLOBAL_SHARDS)

        shard_examples = []
        for _, row in shard_df.iterrows():
            true_label_idxs = extract_active_indices(row['labels'], CLASS_TO_IDX)
            if not true_label_idxs:
                continue
            true_labels = [CORINE_19_CLASSES[i] for i in true_label_idxs]
            shard_examples.extend(build_example(str(row['s1_name']), str(row['patch_id']), true_labels))

        shard_path = os.path.join(DATA_OUTPUT_DIR, f"reben_shard_{shard_idx:05d}.jsonl")
        with open(shard_path, 'w') as f:
            for ex in shard_examples:
                f.write(json.dumps(ex) + "\n")
        print(f"wrote {len(shard_examples):,} judge examples -> {shard_path}")

        with open(PROGRESS_PATH, 'w') as f:
            json.dump({
                "next_shard_idx": shard_idx + 1,
                "total_global_shards": TOTAL_GLOBAL_SHARDS,
                "samples_per_shard": SAMPLES_PER_SHARD,
                "target_bigearth_shard_gb": TARGET_BIGEARTH_SHARD_GB,
                "target_s1_gb": TARGET_S1_GB_PER_SHARD,
                "target_s2_gb": TARGET_S2_GB_PER_SHARD,
                "updated_at": time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
            }, f, indent=2)

        print(f"global shard {shard_idx + 1} complete -- clearing /tmp to free disk space...")
        clear_fusion_tmp_dir()

    print("\n" + "="*75)
    if build_interrupted:
        print(f"BUILD SESSION PAUSED ({reason_for_stop}). Re-run to resume automatically.")
    elif session_end_shard_idx >= TOTAL_GLOBAL_SHARDS:
        print(f"ALL {TOTAL_GLOBAL_SHARDS} GLOBAL SHARDS BUILT INTO VLM TRAINING DATA.")
    else:
        print(f"SESSION COMPLETE: built up to global shard {session_end_shard_idx}/{TOTAL_GLOBAL_SHARDS}.")
        print(f"Next session will automatically resume at global shard {session_end_shard_idx + 1}.")
    print("="*75)


---
## SARLANG-1M stage (from Hugging Face)

Pulls `SARimages_preprocessed.zip` (already despeckled/contrast-stretched --
the version worth using given the earlier finding that this preprocessing
measurably improves VLM accuracy on SAR) and `Text.zip` from
`YiminJimmy/SARLANG-1M`, skipping the much larger `SARimages_original.zip` and
`RGBimages.zip` since they're not needed for this task. Runs once per session
window (guarded by a done-flag), independent of the BigEarthNet shard loop above.

**TODO(adapt):** `Text.zip`'s internal format isn't something I have direct
visibility into -- inspect it after the first download (`!unzip -l` on the
extracted path) and adjust `load_sarlang_text_annotations()` below to match its
actual structure (JSON lines, CSV, or per-image `.txt` files are all common for
datasets like this).


In [ ]:
from huggingface_hub import hf_hub_download

SARLANG_REPO = "YiminJimmy/SARLANG-1M"
SARLANG_DIR = os.path.join(OUTPUT_DIR, "sarlang_data")
SARLANG_DONE_FLAG = os.path.join(SARLANG_DIR, ".sarlang_extracted")
os.makedirs(SARLANG_DIR, exist_ok=True)

elapsed = time.time() - SESSION_START_TIME
if elapsed >= MAX_RUNTIME_SECONDS:
    print(f"skipping SARLANG-1M stage -- already past the {MAX_RUNTIME_HOURS}h session limit this run.")
elif os.path.exists(SARLANG_DONE_FLAG):
    print("SARLANG-1M already downloaded and extracted -- skipping.")
else:
    print("downloading SARLANG-1M (preprocessed SAR images + text annotations)...")
    images_zip = hf_hub_download(repo_id=SARLANG_REPO, repo_type="dataset", filename="SARimages_preprocessed.zip")
    text_zip = hf_hub_download(repo_id=SARLANG_REPO, repo_type="dataset", filename="Text.zip")

    import zipfile
    with zipfile.ZipFile(images_zip, 'r') as zf:
        zf.extractall(os.path.join(SARLANG_DIR, "images"))
    with zipfile.ZipFile(text_zip, 'r') as zf:
        zf.extractall(os.path.join(SARLANG_DIR, "text"))

    with open(SARLANG_DONE_FLAG, 'w') as f:
        f.write('done')
    print("SARLANG-1M downloaded and extracted.")


In [ ]:
def load_sarlang_text_annotations(text_dir: str) -> list[dict]:
    """Loads SARLANG-1M annotation records across common formats (JSON/JSONL/CSV)."""
    records = []
    json_candidates = glob.glob(os.path.join(text_dir, "**", "*.json"), recursive=True)
    jsonl_candidates = glob.glob(os.path.join(text_dir, "**", "*.jsonl"), recursive=True)
    csv_candidates = glob.glob(os.path.join(text_dir, "**", "*.csv"), recursive=True)

    if jsonl_candidates:
        for path in jsonl_candidates:
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        records.append(json.loads(line))
    elif json_candidates:
        for path in json_candidates:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                records.extend(data if isinstance(data, list) else [data])
    elif csv_candidates:
        for path in csv_candidates:
            records.extend(pd.read_csv(path).to_dict(orient="records"))
    else:
        print(f"no recognized annotation format found under {text_dir} -- inspect contents manually.")

    print(f"loaded {len(records):,} raw SARLANG-1M annotation record(s)")
    return records

def normalize_sarlang_records(records: list[dict], images_dir: str) -> list[dict]:
    """Maps raw SARLANG-1M annotation records to standardized conversational VLM format."""
    normalized = []
    for rec in records:
        image_name = rec.get("image") or rec.get("image_path") or rec.get("filename") or rec.get("img_name")
        caption = rec.get("caption") or rec.get("text") or rec.get("description") or rec.get("label")
        if not image_name or not caption:
            continue
        image_path = os.path.join(images_dir, image_name)
        normalized.append({
            "image": image_path,
            "image_name": str(image_name),
            "conversations": [
                {"from": "human", "value": "Describe the land cover, backscatter characteristics, and features visible in this SAR radar image."},
                {"from": "gpt", "value": str(caption)},
            ],
        })
    return normalized

# --- Execute SARLANG-1M Sharding (Target: exactly 7.1 GB per shard) ---
if os.path.exists(SARLANG_DONE_FLAG):
    raw_records = load_sarlang_text_annotations(os.path.join(SARLANG_DIR, "text"))
    images_dir = os.path.join(SARLANG_DIR, "images")
    normalized = normalize_sarlang_records(raw_records, images_dir)

    print(f"\n{'='*70}")
    print(f"📦 Partitioning SARLANG-1M into {TARGET_SARLANG_SHARD_GB} GB shards ({TARGET_SARLANG_SHARD_BYTES:,} bytes each)...")
    print(f"{'='*70}")

    sarlang_shard_idx = 0
    current_shard_records = []
    current_shard_bytes = 0

    for ex in normalized:
        img_p = ex.get("image", "")
        img_bytes = os.path.getsize(img_p) if os.path.exists(img_p) else (200 * 1024)  # ~200 KB fallback
        rec_bytes = len(json.dumps(ex).encode('utf-8')) + img_bytes

        if current_shard_bytes + rec_bytes > TARGET_SARLANG_SHARD_BYTES and current_shard_records:
            shard_path = os.path.join(DATA_OUTPUT_DIR, f"sarlang_shard_{sarlang_shard_idx:05d}.jsonl")
            with open(shard_path, 'w', encoding='utf-8') as f:
                for r in current_shard_records:
                    f.write(json.dumps(r) + "\n")
            shard_gb = current_shard_bytes / (1024**3)
            print(f"   ✅ wrote SARLANG shard {sarlang_shard_idx:05d}: {len(current_shard_records):,} records ({shard_gb:.2f} GB / {TARGET_SARLANG_SHARD_GB} GB target) -> {shard_path}")
            sarlang_shard_idx += 1
            current_shard_records = [ex]
            current_shard_bytes = rec_bytes
        else:
            current_shard_records.append(ex)
            current_shard_bytes += rec_bytes

    # Final remaining shard
    if current_shard_records:
        shard_path = os.path.join(DATA_OUTPUT_DIR, f"sarlang_shard_{sarlang_shard_idx:05d}.jsonl")
        with open(shard_path, 'w', encoding='utf-8') as f:
            for r in current_shard_records:
                f.write(json.dumps(r) + "\n")
        shard_gb = current_shard_bytes / (1024**3)
        print(f"   ✅ wrote SARLANG shard {sarlang_shard_idx:05d}: {len(current_shard_records):,} records ({shard_gb:.2f} GB / {TARGET_SARLANG_SHARD_GB} GB target) -> {shard_path}")
        sarlang_shard_idx += 1

    print(f"\n✨ SARLANG-1M partitioning complete: {sarlang_shard_idx} shard(s) created at {TARGET_SARLANG_SHARD_GB} GB each.")


## Between sessions

1. Save `/kaggle/working/vlm_verification_data` (both `reben_shard_*.jsonl` and
   `sarlang_shard_*.jsonl` files, plus `_build_progress.json`) as a new version
   of a Kaggle Dataset.
2. Attach it as input to the next session -- the auto-detection cell picks it
   up and `next_shard_idx` resumes the BigEarthNet loop where it left off. The
   SARLANG-1M stage's own done-flag prevents re-downloading it every session.
3. Once building is complete, point `DATA_DIR` in
   `sar_vlm_judge_build_and_finetune.ipynb`'s Phase 2 at this same directory
   (with `SKIP_PHASE_1 = True`) to fine-tune on it.
